# 00 — Run all follow-up experiments

Queues `01`–`08` via `scripts/run_all_experiments.py` on one GPU session.

**Before running:** Runtime → GPU (A100 preferred). Expect many hours and Colab compute units.

Skips finished runs (`results/<run>/best.safetensors`). Shuts down the runtime at the end unless you set `NO_SHUTDOWN = True`.

Experiment 08 needs the Cedar Key Figshare LAS under `data/raw/cedar_key_mangrove/` or it is skipped.

In [ ]:
USE_DRIVE = True
REPO_URL = "https://github.com/koshiyomi/pointsam-training.git"
BRANCH = "jianwei-branch"
ONLY = ""  # e.g. "01,02" or "" for all
FORCE = False
NO_SHUTDOWN = False
SKIP_DATA_PREP = False

In [ ]:
import os, sys, subprocess
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    root = Path("/content/drive/MyDrive/pointsam-training")
else:
    root = Path("/content/pointsam-training")

if not (root / "src" / "treeseg").is_dir():
    root.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["git", "clone", "-b", BRANCH, "--depth", "1", REPO_URL, str(root)])

os.chdir(root)
sys.path.insert(0, "src")
print("cwd", os.getcwd())

import torch
assert torch.cuda.is_available(), "Enable a GPU runtime first (Runtime → Change runtime type → GPU)"
print("gpu", torch.cuda.get_device_name(0))

In [ ]:
# One-time env install if needed (uses Drive-cached torkit3d wheel when present)
!python scripts/setup_colab.py --download-data --prepare-data

In [ ]:
cmd = [sys.executable, "scripts/run_all_experiments.py"]
if ONLY.strip():
    cmd += ["--only", ONLY.strip()]
if FORCE:
    cmd.append("--force")
if SKIP_DATA_PREP:
    cmd.append("--skip-data-prep")
if NO_SHUTDOWN:
    cmd.append("--no-shutdown")
print("+", " ".join(cmd), flush=True)
subprocess.check_call(cmd)